# 0D Engine Simulator - Quick Start

This notebook demonstrates the basic usage of the engine simulator for a single-zone HCCI simulation.

## Prerequisites

Make sure you have installed the package:
```bash
pip install -e .
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from engine_sim.simulation.engine import EngineConfig, EngineSimulation

## 1. Load Default Configuration

The simulator ships with a default configuration for a Nissan HCCI engine:
- Bore/Stroke: 86 mm
- Compression Ratio: 12.5:1
- Fuel: iso-octane (C8H18)
- Equivalence ratio: 0.7 (lean)
- EGR: 30%

In [ ]:
config = EngineConfig.from_yaml()

print(f"Fuel: {config.fuel}")
print(f"Bore: {config.bore*1000:.1f} mm")
print(f"Stroke: {config.stroke*1000:.1f} mm")
print(f"Compression Ratio: {config.comp_ratio}")
print(f"Equivalence Ratio: {config.phi}")
print(f"EGR Fraction: {config.egr}")
print(f"Engine Speed: {config.speed:.0f} RPM")
print(f"Initial T: {config.temperature:.0f} K")
print(f"Initial P: {config.pressure/1e5:.2f} bar")

## 2. Run a Single-Zone Simulation

The simulation solves the closed cycle from IVC (-180 CA) to EVO (180 CA).

In [ ]:
sim = EngineSimulation(config)
results = sim.run()

## 3. View Results

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# P-V diagram
axes[0, 0].plot(results.volume * 1e6, results.pressure / 1e5)
axes[0, 0].set_xlabel('Volume [cm\u00b3]')
axes[0, 0].set_ylabel('Pressure [bar]')
axes[0, 0].set_title('P-V Diagram')
axes[0, 0].grid(True, alpha=0.3)

# Temperature
axes[0, 1].plot(results.crank_angle, results.temperature, 'r-')
axes[0, 1].set_xlabel('Crank Angle [deg]')
axes[0, 1].set_ylabel('Temperature [K]')
axes[0, 1].set_title('Temperature')
axes[0, 1].grid(True, alpha=0.3)

# Pressure
axes[1, 0].plot(results.crank_angle, results.pressure / 1e5, 'b-')
axes[1, 0].set_xlabel('Crank Angle [deg]')
axes[1, 0].set_ylabel('Pressure [bar]')
axes[1, 0].set_title('Pressure')
axes[1, 0].grid(True, alpha=0.3)

# Major species
for name in ['C8H18', 'O2', 'CO2', 'H2O']:
    if name in results.species_names:
        idx = results.species_names.index(name)
        axes[1, 1].plot(results.crank_angle, results.species[idx], label=name)
axes[1, 1].set_xlabel('Crank Angle [deg]')
axes[1, 1].set_ylabel('Mass Fraction [-]')
axes[1, 1].set_title('Major Species')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Performance Metrics

In [ ]:
perf = results.calculate_performance()

print(f"Indicated Work:   {perf['indicated_work']:.2f} J")
print(f"IMEP:             {perf['imep']:.2f} bar")
print(f"Peak Pressure:    {perf['peak_pressure']:.2f} bar")
print(f"Peak Temperature: {perf['peak_temperature']:.0f} K")

## 5. Sensitivity: Varying Equivalence Ratio

Let's see how the equivalence ratio affects combustion. We'll compare phi = 0.5, 0.7, and 0.9.

In [ ]:
phi_values = [0.5, 0.7, 0.9]
results_dict = {}

for phi in phi_values:
    cfg = EngineConfig.from_yaml()
    cfg.phi = phi
    sim = EngineSimulation(cfg)
    results_dict[phi] = sim.run()
    perf = results_dict[phi].calculate_performance()
    print(f"phi={phi:.1f}: IMEP={perf['imep']:.2f} bar, "
          f"Tpeak={perf['peak_temperature']:.0f} K, "
          f"Ppeak={perf['peak_pressure']:.1f} bar")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for phi, res in results_dict.items():
    ax1.plot(res.crank_angle, res.temperature, label=f'phi={phi}')
    ax2.plot(res.crank_angle, res.pressure / 1e5, label=f'phi={phi}')

ax1.set_xlabel('Crank Angle [deg]')
ax1.set_ylabel('Temperature [K]')
ax1.set_title('Temperature vs Equivalence Ratio')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Crank Angle [deg]')
ax2.set_ylabel('Pressure [bar]')
ax2.set_title('Pressure vs Equivalence Ratio')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()